In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import timm
from tqdm.notebook import tqdm
import cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_DIR = 'dataset/imgs/test_cropped_v2'
SAMPLE_SUBMISSION_PATH = './dataset/sample_submission.csv'
IMG_SIZE = 384  # 👈 Swin 专属尺寸
BATCH_SIZE = 128
FOLDS = [0, 1, 2, 3, 4]

# 加载测试集路径
df_test = pd.read_csv(SAMPLE_SUBMISSION_PATH)
image_names = df_test['img'].values

# 标准 ImageNet 归一化
transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class TestFeatureDataset(Dataset):
    def __len__(self): return len(image_names)
    def __getitem__(self, idx):
        img_path = os.path.join(TEST_DIR, image_names[idx])
        image = cv2.imread(img_path)
        if image is None:
            image = cv2.imread(os.path.join('./dataset/imgs/test', image_names[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return transform(image=image)['image']

print("🚀 准备提取 Swin-Base 5折骨干特征...")
test_loader = DataLoader(TestFeatureDataset(), batch_size=BATCH_SIZE, shuffle=False, num_workers=8)

all_fold_features = []

for fold in FOLDS:
    print(f"\n👨‍⚖️ 正在加载 Swin Fold {fold} 模型...")
    
    # 1. 每次循环创建一个干净的模型
    model = timm.create_model('swin_base_patch4_window12_384.ms_in22k', pretrained=False, num_classes=10)
    
    # 2. 读取对应折的权重 (注意路径与 BEiT 和 EffB3 不同)
    weight_path = f'models/best_model_swin_fold_{fold}.pth'
    if not os.path.exists(weight_path):
        print(f"⚠️ 找不到权重 {weight_path}，跳过此折！")
        continue
        
    model.load_state_dict(torch.load(weight_path, map_location=DEVICE, weights_only=True))

    # 3. 切除分类头！
    model.reset_classifier(0)
    model.to(DEVICE)
    model.eval()

    # 编译加速
    if int(torch.__version__.split('.')[0]) >= 2:
        model = torch.compile(model)

    fold_features = []
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            for images in tqdm(test_loader, desc=f"💎 提取 Fold {fold} 特征"):
                features = model(images.to(DEVICE))
                fold_features.append(features.cpu().numpy())

    current_fold_final_features = np.concatenate(fold_features, axis=0)
    all_fold_features.append(current_fold_final_features)
    
    del model
    torch.cuda.empty_cache()

if not all_fold_features:
    raise ValueError("🚨 没有成功提取到任何特征！")

# 🎯 对 5 折特征求算术平均
print("\n🤝 正在对 5 折特征进行算术平均融合...")
all_fold_features = np.array(all_fold_features) 
final_avg_features = np.mean(all_fold_features, axis=0) 

save_path = 'models/test_features_swin.npy'
np.save(save_path, final_avg_features)
print(f"✅ Swin 5折特征融合完毕！矩阵形状: {final_avg_features.shape}")
print(f"📁 已安全保存至: {save_path}")

🚀 准备提取 Swin-Base 5折骨干特征...

👨‍⚖️ 正在加载 Swin Fold 0 模型...


💎 提取 Fold 0 特征:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载 Swin Fold 1 模型...


💎 提取 Fold 1 特征:   0%|          | 0/623 [00:00<?, ?it/s]

In [13]:
import os
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm

# ==========================================
# ⚙️ 1. 核心超参数配置区 (供线上疯狂调参)
# ==========================================
BASE_SUBMISSION_PATH = 'models/stacking_ensemble_submission.csv'
OUTPUT_PATH = 'models/final_magic_knn_submission.csv'

# 各个模型提取的测试集高维特征，以及它们对应的权重！
# 必须一一对应！你可以根据它们单模型的得分来分配这里的权重
FEATURE_PATHS = [
    'models/test_features_swin.npy',
    'models/test_features_beit.npy',  
    'models/test_features_effb3.npy'  
]
FEATURE_WEIGHTS = [0.4, 0.35, 0.25] # 👈 新增：特征权重！加起来最好等于1

# KNN 超参数
K = 25              
ALPHA = 5.0         
CLIP_MIN = 1e-5     

# ==========================================
# 📊 2. 数据与特征加载
# ==========================================
print("🔍 [1/5] 正在加载基础预测概率...")
sub_df = pd.read_csv(BASE_SUBMISSION_PATH)
cols = [f'c{i}' for i in range(10)]
original_probs = sub_df[cols].values

print("🧩 [2/5] 正在独立归一化并加权拼接高维视觉特征...")
weighted_features_list = []

for path, weight in zip(FEATURE_PATHS, FEATURE_WEIGHTS):
    if os.path.exists(path):
        print(f"   -> 加载特征: {path} | 权重: {weight}")
        feat = np.load(path)
        
        # 🎯 魔法 1：先独立进行 L2 归一化，大家回到同一起跑线
        feat_norm = normalize(feat, norm='l2', axis=1)
        
        # 🎯 魔法 2：乘上你赋予它的权重
        feat_weighted = feat_norm * weight
        
        weighted_features_list.append(feat_weighted)
    else:
        print(f"   ⚠️ 警告: 找不到特征文件 {path}，已跳过。")

if not weighted_features_list:
    raise ValueError("🚨 没有找到任何特征文件，请检查路径！")

# 拼接加权后的特征
combined_features = np.hstack(weighted_features_list)

# 🎯 魔法 3：整体拼接后再做一次 L2 归一化，喂给 KNN
print("⚖️ [3/5] 执行全局特征 L2 归一化...")
combined_features = normalize(combined_features, norm='l2', axis=1)

# ==========================================
# 🌲 3. 构建 KNN 索引树
# ==========================================
print(f"🌲 [4/5] 正在构建 KNN 图结构 (K={K}, Metric=Cosine)...")
# n_jobs=-1 拉满 CPU 算力
knn = NearestNeighbors(n_neighbors=K, metric='cosine', n_jobs=-1)
knn.fit(combined_features)

print("🔎 正在计算测试集中每一张图的近邻边界...")
# distances 返回的是余弦距离 (1 - cos_sim)
distances, indices = knn.kneighbors(combined_features)

# ==========================================
# 🤝 4. 距离加权的概率平滑 (Message Passing)
# ==========================================
print(f"🤝 [5/5] 执行非线性距离加权平滑 (Alpha={ALPHA})...")
smoothed_probs = np.zeros_like(original_probs)

for i in tqdm(range(len(original_probs)), desc="Graph Smoothing"):
    neighbor_idx = indices[i]
    neighbor_dist = distances[i] 
    
    # 将余弦距离转换为余弦相似度 (值在 0~1 之间，越大越相似)
    # 加上 1e-8 防止除以 0 的计算溢出
    similarities = 1.0 - neighbor_dist 
    similarities = np.clip(similarities, 0.0, 1.0)
    
    # 核心魔法：使用指数 alpha 放大极度相似的帧的权重
    weights = similarities ** ALPHA
    
    # 权重归一化 (让当前节点周围的邻居权重和为 1)
    weights = weights / (weights.sum() + 1e-8)
    
    # 加权聚合 (相当于 GCN 中的 Aggregate 步骤)
    smoothed_probs[i] = np.average(original_probs[neighbor_idx], axis=0, weights=weights)

# ==========================================
# 🛡️ 5. Log Loss 极限防爆与保存
# ==========================================
print(f"🛡️ 执行概率裁剪 (Clip [{CLIP_MIN}, {1.0 - CLIP_MIN}])...")
smoothed_probs = np.clip(smoothed_probs, CLIP_MIN, 1.0 - CLIP_MIN)

# 裁剪后，概率和可能不等于 1，必须再次归一化
smoothed_probs = smoothed_probs / smoothed_probs.sum(axis=1, keepdims=True)

# 写入 DataFrame
sub_df[cols] = smoothed_probs

sub_df.to_csv(OUTPUT_PATH, index=False)
print(f"🎉 终极魔法版 CSV 已生成！文件: {OUTPUT_PATH}")
print("🥇 祝刷榜顺利！")

🔍 [1/5] 正在加载基础预测概率...
🧩 [2/5] 正在独立归一化并加权拼接高维视觉特征...
   -> 加载特征: models/test_features_swin.npy | 权重: 0.4
   -> 加载特征: models/test_features_beit.npy | 权重: 0.35
   -> 加载特征: models/test_features_effb3.npy | 权重: 0.25
⚖️ [3/5] 执行全局特征 L2 归一化...
🌲 [4/5] 正在构建 KNN 图结构 (K=25, Metric=Cosine)...
🔎 正在计算测试集中每一张图的近邻边界...
🤝 [5/5] 执行非线性距离加权平滑 (Alpha=3.0)...


Graph Smoothing:   0%|          | 0/79726 [00:00<?, ?it/s]

🛡️ 执行概率裁剪 (Clip [1e-05, 0.99999])...
🎉 终极魔法版 CSV 已生成！文件: models/final_magic_knn_submission.csv
🥇 祝刷榜顺利！
